# Autonomous Navigation Drone — Rubric Completion Addendum

This notebook is a **safe addendum** to complete the missing rubric parts for the course project.

## How to use this file

1. Keep your original notebook as the main implementation file.
2. Run this addendum after the original notebook if you want the optional analysis cells to use the trained variables.
3. The code cells in this addendum are written to be **safe**:
   - If the original variables/files are available, they will run the extra analysis.
   - If they are not available, they will skip gracefully without raising errors.

## Project focus

The project demonstrates practical understanding of:

- CNNs
- RNNs
- LSTMs
- GRUs
- Transformers / Attention
- Pretrained CNN feature extraction using MobileNetV2
- Comparative experimentation
- Error analysis and reflection

# 1. Problem Understanding & Task Framing

## Problem Statement

The goal of this project is to build an autonomous drone navigation prototype that predicts the next navigation action from a short sequence of video frames.

The model receives a sequence of consecutive frames captured from a drone-like navigation video and predicts one of four navigation classes:

| Class ID | Class Name |
|---:|---|
| 0 | Left |
| 1 | Right |
| 2 | Straight |
| 3 | Stop |

## Input and Output

- **Input:** A sequence of 5 consecutive RGB frames.
- **Frame size:** 224 × 224 × 3.
- **Output:** One navigation action: Left, Right, Straight, or Stop.

## Why this is a deep learning problem

Navigation decisions depend on both:

1. **Spatial features** inside each frame, such as brightness, road/scene structure, and visual patterns.
2. **Temporal features** across consecutive frames, because movement direction is easier to understand when we analyze how the scene changes over time.

For this reason, the project compares:

- A CNN baseline using only one frame.
- Sequence models using CNN features over multiple frames.
- Recurrent models: RNN, LSTM, and GRU.
- A Transformer-based model using self-attention over the frame sequence.

# 2. Dataset Construction & Pseudo-Labeling Explanation

## Dataset Source

The navigation dataset is built from extracted frames from a video. Each training example is created by grouping 5 consecutive frames into one sequence.

## Labeling Method

The labels are generated using a rule-based pseudo-labeling heuristic from the final frame of each sequence.

The algorithm compares the average grayscale brightness of the left and right halves of the frame:

- If the whole frame is very dark, the label is **Stop**.
- If the left and right halves are approximately balanced, the label is **Straight**.
- If the left side is darker than the right side, the label is **Right**.
- Otherwise, the label is **Left**.

## Important Limitation

The dataset is not manually annotated by humans. The labels are **pseudo-labels**, meaning they are automatically generated using a heuristic rule.

This is acceptable for a course prototype, but it creates some limitations:

- Some labels may be noisy or incorrect.
- The model may learn the brightness heuristic instead of true navigation behavior.
- Classes may become imbalanced, especially the Straight class.
- Better results would require real drone navigation labels or manual annotation.

In [ ]:
# Safe reproducibility setup
# This cell does not require the original notebook variables.

import os
import random
import numpy as np

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

try:
    import tensorflow as tf
    tf.random.set_seed(SEED)
    print("TensorFlow seed set successfully.")
except Exception as e:
    print("TensorFlow is not available in this runtime. Skipping TensorFlow seed.")
    print("Reason:", str(e))

print("Reproducibility seed:", SEED)

In [ ]:
# Safe dataset summary cell
# If X.npy and y.npy exist, this cell prints dataset statistics.
# If not, it skips without error.

import os
import numpy as np
from collections import Counter

class_names = ['Left', 'Right', 'Straight', 'Stop']

if os.path.exists("X.npy") and os.path.exists("y.npy"):
    X_loaded = np.load("X.npy")
    y_loaded = np.load("y.npy")

    print("Dataset files found.")
    print("X shape:", X_loaded.shape)
    print("y shape:", y_loaded.shape)

    counts = Counter(y_loaded.tolist())
    print("\nClass distribution:")
    for class_id, class_name in enumerate(class_names):
        print(f"{class_id} - {class_name}: {counts.get(class_id, 0)} samples")

    print("\nNote: These labels are pseudo-labels generated from frame brightness rules.")
else:
    print("X.npy and/or y.npy were not found in the current folder.")
    print("No error: run the original notebook first if you want this cell to show dataset statistics.")

# 3. Model Selection & Architectural Justification

The project uses several architectures to compare spatial and temporal learning strategies.

| Model | Why it was selected |
|---|---|
| CNN Baseline | Used as a simple baseline to test whether a single frame is enough for navigation prediction. |
| MobileNetV2 Feature Extractor | A pretrained lightweight CNN is used to extract strong visual features from each frame without training a large CNN from scratch. |
| CNN + SimpleRNN | Tests whether basic recurrent sequence learning improves performance over single-frame CNN. |
| CNN + LSTM | LSTM can keep useful temporal information across frames and reduce the vanishing-gradient problem compared to SimpleRNN. |
| CNN + GRU | GRU is a lighter recurrent architecture than LSTM and can perform well with fewer parameters. |
| CNN + Transformer | Uses self-attention to learn relationships between all frames in the sequence, instead of processing frames strictly step by step. |

## Main modeling idea

Each frame is first converted into a visual feature vector using a CNN feature extractor.  
The sequence of feature vectors is then passed to a temporal model such as RNN, LSTM, GRU, or Transformer.

This design separates the problem into:

1. **Spatial understanding:** What is inside each frame?
2. **Temporal understanding:** How does the visual scene evolve across frames?

# 4. Experimental Results & Comparative Analysis

The original notebook compares five different model families.

| Rank | Model | Accuracy |
|---:|---|---:|
| 1 | CNN + LSTM | 83.13% |
| 1 | CNN + GRU | 83.13% |
| 3 | CNN + RNN | 81.88% |
| 4 | CNN + Transformer | 80.00% |
| 5 | CNN Baseline | 56.88% |

## Interpretation

The CNN baseline performs the worst because it only uses the last frame of the sequence. This means it ignores movement and temporal context.

The sequence models perform much better because they use information from multiple frames. This supports the idea that navigation is a temporal prediction problem.

The best models are CNN + LSTM and CNN + GRU. Both models are designed for sequence learning and can capture short-term motion patterns between consecutive frames.

The Transformer model is strong conceptually, but it does not outperform LSTM/GRU in this experiment. The likely reasons are:

- The dataset is relatively small.
- The sequence length is short.
- The labels are heuristic pseudo-labels.
- There is class imbalance, especially for the Straight class.

In [ ]:
# Safe model comparison table and plot
# This cell uses the reported results from the original notebook.
# It does not depend on trained model variables.

import pandas as pd

results = pd.DataFrame({
    "Model": [
        "CNN + LSTM",
        "CNN + GRU",
        "CNN + RNN",
        "CNN + Transformer",
        "CNN Baseline"
    ],
    "Accuracy (%)": [83.13, 83.13, 81.88, 80.00, 56.88]
})

print("Model Comparison:")
try:
    display(results)
except Exception:
    print(results.to_string(index=False))

try:
    import matplotlib.pyplot as plt

    plt.figure(figsize=(10, 5))
    bars = plt.bar(results["Model"], results["Accuracy (%)"])
    plt.ylabel("Accuracy (%)")
    plt.xlabel("Model")
    plt.title("Navigation Model Comparison")
    plt.ylim(0, 100)
    plt.xticks(rotation=20, ha="right")

    for bar, val in zip(bars, results["Accuracy (%)"]):
        plt.text(
            bar.get_x() + bar.get_width() / 2,
            bar.get_height() + 1,
            f"{val:.2f}%",
            ha="center"
        )

    plt.tight_layout()
    plt.savefig("rubric_addendum_model_comparison.png", dpi=150, bbox_inches="tight")
    plt.show()
except Exception as e:
    print("Plot skipped safely.")
    print("Reason:", str(e))

# 5. Detailed Error Analysis

The Transformer classification report showed an important issue:

| Class | Precision | Recall | F1-score | Support |
|---|---:|---:|---:|---:|
| Left | 0.78 | 0.93 | 0.85 | 58 |
| Right | 0.73 | 0.88 | 0.80 | 43 |
| Straight | 0.00 | 0.00 | 0.00 | 17 |
| Stop | 0.92 | 0.86 | 0.89 | 42 |

## Key observation

The Transformer achieved 80% accuracy, but it completely failed to predict the **Straight** class in the reported test set.

This means accuracy alone is not enough to evaluate the model. The macro F1-score and class-level performance are more informative.

## Why did this happen?

Possible reasons:

1. **Class imbalance:** Straight has only 17 samples in the test set.
2. **Noisy pseudo-labels:** The Straight label is generated when left and right brightness are similar. This rule may be unstable in real frames.
3. **Visual similarity:** Straight examples may look similar to Left or Right frames.
4. **Small dataset:** The Transformer usually needs more data than LSTM/GRU to generalize well.
5. **Short sequence length:** A 5-frame sequence may not provide enough motion information for attention to be clearly useful.

## Improvement plan

To improve the Straight class:

- Collect or annotate more Straight samples.
- Use class weights during training.
- Oversample minority classes.
- Use better labels from real drone control commands.
- Add optical flow or motion vectors.
- Evaluate models using macro F1-score, not accuracy only.
- Compare confusion matrices for the best model, not only the Transformer.

In [ ]:
# Safe confusion matrix and classification report for the best available trained model
# If the original trained variables exist, this cell evaluates the best model.
# If they do not exist, it skips without error.

def safe_best_model_error_analysis():
    required = ["X_test_feat", "y_test", "class_names"]
    missing = [name for name in required if name not in globals()]

    if missing:
        print("Skipping live model error analysis because these variables are missing:", missing)
        print("No error: run this cell after the original notebook if you want live evaluation.")
        return

    import numpy as np

    candidate_models = [
        ("CNN + LSTM", globals().get("model_lstm"), globals().get("lstm_acc", None)),
        ("CNN + GRU", globals().get("model_gru"), globals().get("gru_acc", None)),
        ("CNN + RNN", globals().get("model_rnn"), globals().get("rnn_acc", None)),
        ("CNN + Transformer", globals().get("model_transformer"), globals().get("transformer_acc", None)),
    ]

    available = [(name, model, acc) for name, model, acc in candidate_models if model is not None]

    if not available:
        print("No trained sequence model was found in memory.")
        print("No error: run the original notebook first, then run this cell.")
        return

    available_with_acc = [item for item in available if item[2] is not None]
    if available_with_acc:
        best_name, best_model, best_acc = sorted(available_with_acc, key=lambda x: x[2], reverse=True)[0]
    else:
        best_name, best_model, best_acc = available[0]

    print(f"Best available model selected: {best_name}")
    if best_acc is not None:
        print(f"Reported accuracy: {best_acc * 100:.2f}%")

    try:
        from sklearn.metrics import classification_report, confusion_matrix
        import matplotlib.pyplot as plt

        y_pred = best_model.predict(X_test_feat, verbose=0)
        y_pred_classes = np.argmax(y_pred, axis=1)
        y_true_classes = np.argmax(y_test, axis=1)

        print("\nClassification Report:")
        print(classification_report(
            y_true_classes,
            y_pred_classes,
            target_names=class_names,
            zero_division=0
        ))

        cm = confusion_matrix(y_true_classes, y_pred_classes)

        plt.figure(figsize=(7, 5))
        plt.imshow(cm)
        plt.title(f"{best_name} - Confusion Matrix")
        plt.xlabel("Predicted Label")
        plt.ylabel("True Label")
        plt.xticks(range(len(class_names)), class_names, rotation=30)
        plt.yticks(range(len(class_names)), class_names)

        for i in range(cm.shape[0]):
            for j in range(cm.shape[1]):
                plt.text(j, i, str(cm[i, j]), ha="center", va="center")

        plt.tight_layout()
        plt.savefig("best_model_confusion_matrix.png", dpi=150, bbox_inches="tight")
        plt.show()

    except Exception as e:
        print("Live error analysis skipped safely.")
        print("Reason:", str(e))

safe_best_model_error_analysis()

# 6. Innovation & Intellectual Contribution

The project is not only a single baseline model. It includes several meaningful extensions:

1. **Multi-model comparison:** The project compares CNN, RNN, LSTM, GRU, and Transformer-based strategies.
2. **Pretrained feature extraction:** MobileNetV2 is used as a lightweight pretrained CNN backbone.
3. **Temporal navigation framing:** The problem is framed as sequence prediction, not simple image classification.
4. **Drone information analyzer:** The project includes an additional CNN-based multi-output model that predicts drone type, speed category, and price range.
5. **Practical reflection:** The project identifies limitations such as pseudo-label quality, class imbalance, and failure cases.

## Important clarification about the Drone Info System

The Drone Info System should be presented as an experimental extension or proof-of-concept.

The original dataset scraping produced only a small number of valid real images. Therefore, the notebook generated synthetic images as a fallback to demonstrate the idea of multi-output CNN classification.

This part should not be presented as a production-ready drone classifier. It is a demonstration of the architecture and training pipeline.

# 7. Limitations & Reflection

## Main limitations

1. **Pseudo-labeling limitation:** The labels are created automatically using brightness rules instead of human annotation.
2. **Class imbalance:** The Straight class has fewer samples and weaker performance.
3. **Small dataset:** The dataset size is limited, which affects generalization.
4. **Short temporal window:** A sequence length of 5 may not capture enough movement in all cases.
5. **No real flight-control labels:** The project does not use actual drone steering commands.
6. **Synthetic fallback in drone info system:** The extra drone information module uses synthetic data when real image downloads are insufficient.

## What I learned

This project shows that temporal deep learning models can improve navigation prediction compared to a single-frame CNN baseline.

It also shows that a high accuracy score can hide serious class-level failures. Therefore, classification reports, confusion matrices, and macro F1-score are important for evaluating real model behavior.

## Future improvements

- Use real annotated drone navigation data.
- Add optical flow or motion features.
- Use class weighting or balanced sampling.
- Increase dataset size.
- Tune hyperparameters.
- Try ViT or TimeSformer-style video models.
- Build a more complete Streamlit demo using the saved best model.

# 8. Reproducibility & Notebook Quality

To make the project reproducible, the following points should be clearly mentioned in the final submission:

| Item | Value / Explanation |
|---|---|
| Random seed | 42 |
| Input frame size | 224 × 224 × 3 |
| Sequence length | 5 frames |
| Classes | Left, Right, Straight, Stop |
| Train/test split | 80% train, 20% test |
| Main pretrained model | MobileNetV2 |
| Baseline | CNN using the last frame only |
| Sequence models | SimpleRNN, LSTM, GRU, Transformer |
| Main metric | Accuracy |
| Additional metrics | Precision, Recall, F1-score, Confusion Matrix |
| Main limitation | Pseudo-labeling and class imbalance |

## Recommended final submission structure

1. Problem statement
2. Dataset collection and pseudo-labeling
3. Preprocessing pipeline
4. Model architectures
5. Training setup
6. Experimental comparison
7. Error analysis
8. Limitations
9. Future work
10. Conclusion

In [ ]:
# Optional: create a safe app.py file to prevent Streamlit errors
# This creates a simple demo dashboard that does not crash even if trained models are not available.

app_code = 'import streamlit as st\nimport pandas as pd\n\nst.set_page_config(page_title=\'Autonomous Navigation Drone\', layout=\'wide\')\n\nst.title(\'Autonomous Navigation Drone - Course Project Demo\')\n\nst.markdown("""\nThis dashboard summarizes the project results and provides a safe demo interface.\n\nThe main project compares:\n- CNN Baseline\n- CNN + RNN\n- CNN + LSTM\n- CNN + GRU\n- CNN + Transformer\n""")\n\nresults = pd.DataFrame({\n    \'Model\': [\n        \'CNN + LSTM\',\n        \'CNN + GRU\',\n        \'CNN + RNN\',\n        \'CNN + Transformer\',\n        \'CNN Baseline\'\n    ],\n    \'Accuracy (%)\': [83.13, 83.13, 81.88, 80.00, 56.88]\n})\n\nst.subheader(\'Model Comparison\')\nst.dataframe(results, use_container_width=True)\n\nst.subheader(\'Project Notes\')\nst.write("""\nThe best results were achieved by CNN + LSTM and CNN + GRU.\nThe Transformer model was useful for demonstrating attention, but it had weak performance on the Straight class.\nThis shows why class-level evaluation is important.\n""")\n\nst.subheader(\'Upload a Frame or Drone Image\')\nuploaded_file = st.file_uploader(\'Upload an image for demo preview\', type=[\'jpg\', \'jpeg\', \'png\'])\n\nif uploaded_file is not None:\n    st.image(uploaded_file, caption=\'Uploaded image\', use_column_width=True)\n    st.info(\'This safe demo shows the uploaded image. Model prediction can be connected after loading the saved trained model.\')\nelse:\n    st.info(\'Upload an image to preview it here.\')\n'

with open("app.py", "w", encoding="utf-8") as f:
    f.write(app_code)

print("Safe app.py created successfully.")
print("You can run it with: streamlit run app.py")


# 9. Final Conclusion

This project successfully demonstrates a deep learning approach for autonomous drone navigation using both spatial and temporal modeling.

The comparison shows that sequence-based models outperform a single-frame CNN baseline. CNN + LSTM and CNN + GRU achieved the best reported accuracy, which supports the idea that temporal information is important for navigation.

The project also includes a Transformer model to demonstrate attention-based sequence modeling. Although it achieved reasonable overall accuracy, the error analysis showed that it failed on the Straight class. This is an important learning point because it proves that accuracy alone is not enough.

Overall, the project satisfies the course rubric by covering:

- Problem framing
- Model selection and justification
- Correct implementation
- CNN, RNN, LSTM, GRU, Transformer, and pretrained models
- Comparative experimentation
- Error analysis
- Reflection and future improvements

# 10. Doctor Discussion Notes

## If the doctor asks: Why did you use sequence models?

Because navigation depends on movement across frames. A single image may not show whether the drone should go left, right, or straight. Sequence models such as LSTM and GRU can learn temporal patterns across consecutive frames.

## If the doctor asks: Why did CNN baseline perform worse?

The CNN baseline uses only the last frame. It ignores temporal context. That is why it achieved lower accuracy than CNN + LSTM and CNN + GRU.

## If the doctor asks: Why did Transformer not become the best?

The Transformer usually needs more data and stronger labels. In this project, the dataset is small, the sequence length is short, and the labels are pseudo-labels. So LSTM and GRU were more suitable.

## If the doctor asks: What is the biggest weakness of the project?

The biggest weakness is the pseudo-labeling method. The labels are generated using brightness rules, not human annotation or real drone control commands.

## If the doctor asks: How would you improve it?

I would collect real drone navigation data with actual control labels, balance the dataset, add optical flow, and evaluate with macro F1-score as well as accuracy.

## If the doctor asks: What is the innovation?

The project compares multiple sequence learning strategies for navigation, uses a pretrained CNN backbone, includes a Transformer attention model, and adds a drone information analyzer as an experimental extension.